In [2]:
import os

In [ ]:
# OPENAI_API_KEY is read from the environment / .env (see .env.example)

In [4]:
from langchain.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI
from langchain.document_loaders import DirectoryLoader
from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
loader=PyPDFLoader("BNS Delhi police 2nd edition.pdf")
text=loader.load()
text

In [57]:
len(text)

212

In [ ]:
loader=PyPDFLoader("Indian Penal Code Book.pdf")
text+=loader.load()
len(text)

In [ ]:
loader=DirectoryLoader("",glob="ipctextcorpus.txt",loader_cls=TextLoader)
document=loader.load()
document=document+text

In [69]:
len(document)

2084

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
text=text_splitter.split_documents(document)
text

In [74]:
text[200]

Document(page_content='private, whether run by the Central Government, the State Government, local bodies or any other person, \ncontravenes the provisions of section 357C of the Code of Crim inal Procedure, 1973 (2 of 1974), shall be \npunished with imprisonment for a term which may extend to one year or with fine or with both.]  \n167. Public servant framing an incorrect documen t with intent to cause injury .—Whoever, being \na public servant, and be ing, as 2[such public servant, charged with the preparation or translation of any \ndocument  or electronic record , frames , prepares  or translates that document  or electronic record]  in a \nmanner which he knows or believes to be incorrect, intending thereby to cause or knowing it to be likely \nthat he may thereby cause injury to any person, shall be punished with imprisonment of either description \nfor a term which may extend to three ye ars, or with fine, or with both.', metadata={'source': 'ipctextcorpus.txt'})

# chroma db

In [6]:
persist_directory='ip+bns_pdf'

In [8]:
embedding = OpenAIEmbeddings()

In [51]:
print(embedding.model)

text-embedding-ada-002


In [79]:
vectordb=Chroma.from_documents(documents=text,
                               embedding=embedding,
                               persist_directory=persist_directory)

In [80]:
vectordb.persist()
vectordb=None

/Users/aasarmehdi/third year law project/langchain/krishnyk/venv/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  warn_deprecated(


In [9]:
vectordb=Chroma(persist_directory=persist_directory,
                embedding_function=embedding)

# Make a retriever

In [15]:
retriver=vectordb.as_retriever()

In [27]:
docs=retriver.get_relevant_documents("A person's parents found a marriage match on a matrimony site. They connected, and the boy bought an engagement ring. The relationship included a physical component, but the engagement was later broken.")

In [28]:
docs

[Document(page_content='the accused had actually wanted to marry the victim, or had mala ﬁde motives, and had\nmade a false promise to this effect only to satisfy his lust, as the latter falls within the\nambit of cheating or deception. There is a distinction between the mere breach of a\npromise, and not fulﬁlling a false promise. Thus, the Court must examine whether there\nwas made, at an early stage a false promise of marriage by the accused; and whether\nthe consent involved was given after wholly, understanding the nature and', metadata={'page': 295, 'source': 'Indian Penal Code Book.pdf'}),
 Document(page_content="II. He was sentenced to three years RI.46.\n[s 79.8] English case.—\nThe accused took an unmarried girl under the age of 16 years out of the possession,\nand against the will, of her father. The defence of the accused was that he was bona\nﬁde and reasonably believed that the girl was older than 16. It was held that the taking\nof the girl was unlawful the defence was b

# Make a Chain

In [18]:
# from langchain_community.llms import Ollama
## Load Ollama LAMA2 LLM model
llm=OpenAI(temperature=0.3,openai_api_key=os.environ["OPENAI_API_KEY"])
llm

OpenAI(client=<openai.resources.completions.Completions object at 0x12464cac0>, async_client=<openai.resources.completions.AsyncCompletions object at 0x124e885e0>, temperature=0.3, openai_api_key=SecretStr('**********'), openai_proxy='')

In [50]:
print(llm.model_name)

gpt-3.5-turbo-instruct


In [32]:
## Design ChatPrompt Template
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""You are ChatGPT (Legal Assistant – India), specializing in Indian criminal law (IPC/BNS).

You must answer ONLY from the following retrieved context chunks.  
If the answer is not in the context, reply exactly: **"I don't know."**  
Do NOT use outside knowledge.  

---
Question: {input}
                                                                                
# Retrieved Context Chunks:
<context>
{context}
</context>


---

# Important Rules for Answering:
1. **Strict Priority**: If any chunk’s `source` is **"BNS Delhi police 2nd edition.pdf"**, build the answer **primarily from those BNS chunks**.
   - Use IPC sources (`"ipctextcorpus.txt"`, `"Indian Penal Code Book.pdf"`) **only to fill gaps** not covered by the BNS chunks.
   - If BNS and another source conflict, prefer the BNS wording.
2. **Quoting**: Include as much text as possible from the retrieved context.  
   Wrap quotes in double quotes and cite the source and page/chunk_id like:  
   `[BNS Delhi police 2nd edition.pdf, p. 45]` or `[Indian Penal Code Book.pdf, chunk 12]`.
3. **No Fabrication**: Do not invent or infer sections, punishments, or procedures not present in the retrieved text.  
   If insufficient information exists → reply **"I don't know."**
4. **Clarity**: Explanations must be clear, concise, and tailored to the user’s question.

# Answer Format:
- Direct answer to the user’s question, grounded in the BNS chunks first.  
- Supporting explanation with **quoted or closely paraphrased text** from the context.  
- Citations after each paragraph.  
- If nothing is found: **"I don't know."**



"""
)
from langchain.chains.combine_documents import create_stuff_documents_chain

document_chain=create_stuff_documents_chain(llm,prompt)

retriever=vectordb.as_retriever(search_kwargs={"k":10})

from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)


In [33]:
response=retrieval_chain.invoke({"input":"A person's parents found a marriage match on a matrimony site. They connected, and the boy bought an engagement ring. The relationship included a physical component, but the engagement was later broken."})
response

{'input': "A person's parents found a marriage match on a matrimony site. They connected, and the boy bought an engagement ring. The relationship included a physical component, but the engagement was later broken.",
 'context': [Document(page_content='the accused had actually wanted to marry the victim, or had mala ﬁde motives, and had\nmade a false promise to this effect only to satisfy his lust, as the latter falls within the\nambit of cheating or deception. There is a distinction between the mere breach of a\npromise, and not fulﬁlling a false promise. Thus, the Court must examine whether there\nwas made, at an early stage a false promise of marriage by the accused; and whether\nthe consent involved was given after wholly, understanding the nature and', metadata={'page': 295, 'source': 'Indian Penal Code Book.pdf'}),
  Document(page_content="II. He was sentenced to three years RI.46.\n[s 79.8] English case.—\nThe accused took an unmarried girl under the age of 16 years out of the po

In [34]:
response['answer']

'\nThe relationship between the boy and the girl in this scenario involved a physical component and the boy had even bought an engagement ring. However, the engagement was later broken. In this case, it is important to determine whether the boy had made a false promise of marriage to the girl in order to satisfy his lust, which would fall under the ambit of cheating or deception. This distinction is important as the Court must examine whether the consent given by the girl was based on a false promise of marriage or if it was given after fully understanding the nature and consequences of sexual indulgence. [BNS Delhi police 2nd edition.pdf, p. 45]\n\nIn cases where a person makes a false promise of marriage to satisfy their lust, it is considered cheating or deception. This is different from a mere breach of promise and must be examined by the Court. It is important to determine whether the accused had the intention to marry the victim or if they had mala fide motives. [BNS Delhi police

In [35]:
response=retrieval_chain.invoke({"input":"Rioting, armed with deadly weapon?"})

In [36]:
response['answer']

'\nYes, rioting while being armed with a deadly weapon is a punishable offense under the Indian Penal Code. According to section 148, "Whoever is guilty of rioting, being armed with a deadly weapon or with anything which, used as a weapon of offence, is likely to cause death, shall be punished with imprisonment of either description for a term which may extend to three years, or with fine, or with both." [Indian Penal Code Book.pdf, chunk 12]\n\nThis means that if a person participates in a riot while being armed with a deadly weapon or anything that can be used as a weapon to cause harm, they can face imprisonment for up to three years, or a fine, or both. This is considered an aggravated form of the offense of rioting and carries a higher penalty due to the increased risk to public safety. [Indian Penal Code Book.pdf, chunk 12]\n\nFurthermore, section 191(3) states that if the person uses force or violence with a deadly weapon in the riot, they can face imprisonment for up to five ye

In [2]:
import pandas as pd
df=pd.read_csv("legal_qav.csv")

In [3]:
df

,question,answer,llm_answer,contexts
0,A person's parents found a marriage match on a...,The user is advised to go to the nearest polic...,NaN,NaN
1,The police filed a 'B Report' in a case involv...,The response to this question appears to be mi...,NaN,NaN
2,Legal actions against a narcissistic father.,The advice states that a narcissistic personal...,NaN,NaN
3,"A user's uncle, who has been in the Army for a...",The response clarifies that no person is above...,NaN,NaN
4,A user needs legal help to punish two people w...,The legal advice states that offenses related ...,NaN,NaN
...,...,...,...,...
417,What is the punishment if the person the publi...,The public servant can be punished with impris...,NaN,NaN
418,What is the punishment for intentionally insul...,The punishment is simple imprisonment for a te...,NaN,NaN
419,What happens to a person who was sentenced and...,The person receives the punishment to which th...,NaN,NaN
420,What is the punishment for criminal breach of ...,The punishment for criminal breach of trust is...,NaN,NaN


In [4]:
df["question"][5:]

5      The question is not fully visible but pertains...
6      Society members are causing a nuisance on a us...
7      A user was in a committed relationship with a ...
8      A user filed a case under Section 107 CRPC aga...
9      A user was in a relationship for 12 to 15 year...
                             ...                        
417    What is the punishment if the person the publi...
418    What is the punishment for intentionally insul...
419    What happens to a person who was sentenced and...
420    What is the punishment for criminal breach of ...
421    Is the punishment for obstruction separate fro...
Name: question, Length: 417, dtype: object

In [5]:
a=df["question"]

In [46]:
j=0
for i in a:
    response=retrieval_chain.invoke({"input":i})
    df['llm_answer'][j]=response['answer']
    df['contexts'][j]=response['context']
    j+=1

/var/folders/__/flq7sb_j7l93199xmqjbt3fh0000gn/T/ipykernel_7901/256958729.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['llm_answer'][j]=response['answer']
/var/folders/__/flq7sb_j7l93199xmqjbt3fh0000gn/T/ipykernel_7901/256958729.py

In [47]:
df

,question,answer,llm_answer,contexts
0,A person's parents found a marriage match on a...,The user is advised to go to the nearest polic...,"I am ChatGPT, a legal assistant specializing i...",[page_content='the accused had actually wanted...
1,The police filed a 'B Report' in a case involv...,The response to this question appears to be mi...,The enquiry in this case should be conducted b...,[page_content='BNS Handbook Delhi Police Acade...
2,Legal actions against a narcissistic father.,The advice states that a narcissistic personal...,\nI am not able to provide a direct answer to ...,[page_content='and bring about the appropriate...
3,"A user's uncle, who has been in the Army for a...",The response clarifies that no person is above...,\nI don't know.,[page_content='THE INDIAN PENAL CODE\nCHAPTER ...
4,A user needs legal help to punish two people w...,The legal advice states that offenses related ...,"\nI am ChatGPT, a legal assistant specializing...",[page_content='the subsequent marriage is cont...
...,...,...,...,...
417,What is the punishment if the person the publi...,The public servant can be punished with impris...,The punishment for a public servant who fails ...,"[page_content='or whenever a public servant, i..."
418,What is the punishment for intentionally insul...,The punishment is simple imprisonment for a te...,The punishment for intentionally insulting or ...,[page_content='Whoever intentionally offers an...
419,What happens to a person who was sentenced and...,The person receives the punishment to which th...,\nIf a person who was sentenced and given remi...,[page_content='the same act of violation of co...
420,What is the punishment for criminal breach of ...,The punishment for criminal breach of trust is...,"""The punishment for criminal breach of trust i...",[page_content='THE INDIAN PENAL CODE\nCHAPTER ...


In [48]:
df.to_csv("llm_answer.csv")

In [124]:
df

,question,ground_truth,answer,contexts
0,Can someone build a house on land assigned for...,"In a residential colony, if a place is assigne...","Based on the provided context, I must state th...",[page_content='the Legislature in framing sect...
1,Hi sir/Madam My husband always give me mentall...,When both of you and your sons are interested ...,"I can't provide legal advice, but I can offer ...",[page_content='were persistently torturing her...
2,"Sir, I worked in a RMC company as a third part...","Dear client, first contact the employer and as...",I don't see any relevant information regarding...,[page_content='Where a conviction under sectio...
3,Someone hit my car from behind but I moved awa...,"It may be noted that currently, hit-and-run ca...","Based on the context provided, it seems that y...",[page_content='truck against the cyclist and t...
4,What is a bail? What is anticipatory bail? How...,Bail is a release of person who is accused of ...,Given the context of the Indian Penal Code (IP...,"[page_content='without sureties, fails without..."
5,My wife has filed a false divorce case against...,As far as your father and sisters are concerne...,I cannot provide legal advice. If you are expe...,[page_content='an offence under this section.3...
6,How can file a Complaint in Consumer Court on ...,"This case do not attract Consumer Forum, & it'...",I don't know.,[page_content='said ﬁxed deposit adjusted a pa...
7,what can be done legally if someone is not ret...,suit for recovery can be filed.,"Based on the provided context, I'd like to add...","[page_content='take place. The Court has, howe..."
8,can a sister have any claim on brothers salary...,In case the assets are joint and not divided s...,I don't know.\n\nThe provided context does not...,[page_content='wherefrom the ingredients of an...
9,if Father is living and can a daughter demand ...,if it is an ancestral property then yes and if...,I don't know.\n\nThe provided context discusse...,[page_content='list of ornaments and other hou...


In [125]:
df.to_csv("final_ollama3.1_only_llm.csv")

In [1]:
import os
# OPENAI_API_KEY is read from the environment / .env (see .env.example)
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI
# import streamlit as st
# from deep_translator import GoogleTranslator
# translator = GoogleTranslator(source='auto', target='ur')

persist_directory='ip+bns_pdf'
# from langchain_community.embeddings import OllamaEmbeddings
# embedding = OllamaEmbeddings()
embedding = OpenAIEmbeddings()

vectordb=Chroma(persist_directory=persist_directory,
                embedding_function=embedding)
from langchain_community.llms import Ollama
## Load Ollama LAMA2 LLM model
# llm = OpenAI(temperature=0.3)
llm =Ollama(model="llama3.1")

## Design ChatPrompt Template
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""You are ChatGPT (Legal Assistant – India), specializing in Indian criminal law (IPC/BNS).

You must answer ONLY from the following retrieved context chunks.  
If the answer is not in the context, reply exactly: **"I don't know."**  
Do NOT use outside knowledge.  

---
Question: {input}
                                                                                
# Retrieved Context Chunks:
<context>
{context}
</context>


---

# Important Rules for Answering:
1. **Strict Priority**: If any chunk’s `source` is **"BNS Delhi police 2nd edition.pdf"**, build the answer **primarily from those BNS chunks**.
   - Use IPC sources (`"ipctextcorpus.txt"`, `"Indian Penal Code Book.pdf"`) **only to fill gaps** not covered by the BNS chunks.
   - If BNS and another source conflict, prefer the BNS wording.
2. **Quoting**: Include as much text as possible from the retrieved context.  
   Wrap quotes in double quotes and cite the source and page/chunk_id like:  
   `[BNS Delhi police 2nd edition.pdf, p. 45]` or `[Indian Penal Code Book.pdf, chunk 12]`.
3. **No Fabrication**: Do not invent or infer sections, punishments, or procedures not present in the retrieved text.  
   If insufficient information exists → reply **"I don't know."**
4. **Clarity**: Explanations must be clear, concise, and tailored to the user’s question.

# Answer Format:
- Direct answer to the user’s question, grounded in the BNS chunks first.  
- Supporting explanation with **quoted or closely paraphrased text** from the context.  
- Citations after each paragraph.  
- If nothing is found: **"I don't know."**



"""
)
from langchain.chains.combine_documents import create_stuff_documents_chain

document_chain=create_stuff_documents_chain(llm,prompt)

retriever=vectordb.as_retriever()

from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)

response=retrieval_chain.invoke({"input":"What is a bail? What is anticipatory bail? How do I get a bail? How much I have to pay for a bail? Is that amount refundable? Can I take bail in advance if I know somebody is going to take a legal action against me (I did not do anything wrong)? How long a bail is valid? Does it have some expiry?"})
response


⚠️ It looks like you upgraded from a version below 0.6 and could benefit from vacuuming your database. Run chromadb utils vacuum --help for more information.


{'input': 'What is a bail? What is anticipatory bail? How do I get a bail? How much I have to pay for a bail? Is that amount refundable? Can I take bail in advance if I know somebody is going to take a legal action against me (I did not do anything wrong)? How long a bail is valid? Does it have some expiry?',
 'context': [Document(metadata={'page': 30, 'source': 'BNS Delhi police 2nd edition.pdf'}, page_content='Non-Cognizable BailableImprisonment for 2 \nyears, or fine, or both\n178 213 Refusing oath or affirmation when \nduly required by public servant to \nmake it.\nNon-Cognizable BailableSimple imprisonment \nfor 6 months,  or fine \nof 5,000 rupees, or \nboth\n179 214 Refusing to answer public servant \nauthorised to question.\nNon-Cognizable BailableSimple imprisonment \nfor 6 months,  or fine \nof 5,000 rupees, or \nboth\n180 215 Refusing to sign statement made \nto a public servant when legally \nrequired to do so\nNon-Cognizable BailableSimple imprisonment \nfor 3 months,  or 

In [6]:
j=0
for i in a:
    response=retrieval_chain.invoke({"input":i})
    df['llm_answer'][j]=response['answer']
    df['contexts'][j]=response['context']
    j+=1

/var/folders/__/flq7sb_j7l93199xmqjbt3fh0000gn/T/ipykernel_79061/256958729.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['llm_answer'][j]=response['answer']
/var/folders/__/flq7sb_j7l93199xmqjbt3fh0000gn/T/ipykernel_79061/256958729.

In [7]:
df

,question,answer,llm_answer,contexts
0,A person's parents found a marriage match on a...,The user is advised to go to the nearest polic...,"A young couple met on a matrimony site, and th...",[page_content='the accused had actually wanted...
1,The police filed a 'B Report' in a case involv...,The response to this question appears to be mi...,The enquiry in a case involving forgery and mi...,[page_content='BNS Handbook Delhi Police Acade...
2,Legal actions against a narcissistic father.,The advice states that a narcissistic personal...,Given the retrieved context chunks and the imp...,[page_content='and bring about the appropriate...
3,"A user's uncle, who has been in the Army for a...",The response clarifies that no person is above...,"A user's uncle, who has been in the Army for a...",[page_content='THE INDIAN PENAL CODE\nCHAPTER ...
4,A user needs legal help to punish two people w...,The legal advice states that offenses related ...,"Given the situation, it appears that the black...",[page_content='the subsequent marriage is cont...
...,...,...,...,...
417,What is the punishment if the person the publi...,The public servant can be punished with impris...,The punishment for failing to apprehend a pers...,"[page_content='or whenever a public servant, i..."
418,What is the punishment for intentionally insul...,The punishment is simple imprisonment for a te...,The punishment for intentionally insulting or ...,[page_content='Whoever intentionally offers an...
419,What happens to a person who was sentenced and...,The person receives the punishment to which th...,If a person who was sentenced and given remiss...,[page_content='the same act of violation of co...
420,What is the punishment for criminal breach of ...,The punishment for criminal breach of trust is...,The punishment for criminal breach of trust is...,[page_content='THE INDIAN PENAL CODE\nCHAPTER ...


In [8]:
df.to_csv("llm_answer_lamma3.1.csv")

# gemni

In [1]:
import pandas as pd
df=pd.read_csv("legal_qav.csv")

In [2]:
df

,question,answer,llm_answer,contexts
0,A person's parents found a marriage match on a...,The user is advised to go to the nearest polic...,NaN,NaN
1,The police filed a 'B Report' in a case involv...,The response to this question appears to be mi...,NaN,NaN
2,Legal actions against a narcissistic father.,The advice states that a narcissistic personal...,NaN,NaN
3,"A user's uncle, who has been in the Army for a...",The response clarifies that no person is above...,NaN,NaN
4,A user needs legal help to punish two people w...,The legal advice states that offenses related ...,NaN,NaN
...,...,...,...,...
417,What is the punishment if the person the publi...,The public servant can be punished with impris...,NaN,NaN
418,What is the punishment for intentionally insul...,The punishment is simple imprisonment for a te...,NaN,NaN
419,What happens to a person who was sentenced and...,The person receives the punishment to which th...,NaN,NaN
420,What is the punishment for criminal breach of ...,The punishment for criminal breach of trust is...,NaN,NaN


In [ ]:
import os
# OPENAI_API_KEY is read from the environment / .env (see .env.example)

# GOOGLE_API_KEY is read from the environment / .env (see .env.example)
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma



persist_directory='ip+bns_pdf'
# from langchain_community.embeddings import OllamaEmbeddings
# embedding = OllamaEmbeddings()
embedding = OpenAIEmbeddings()

vectordb=Chroma(persist_directory=persist_directory,
                embedding_function=embedding)
from langchain_community.llms import Ollama
## Load Ollama LAMA2 LLM model
# llm = OpenAI(temperature=0.3)
# llm =Ollama(model="llama3.1")
llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0.3)

## Design ChatPrompt Template
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""You are ChatGPT (Legal Assistant – India), specializing in Indian criminal law (IPC/BNS).

You must answer ONLY from the following retrieved context chunks.  
If the answer is not in the context, reply exactly: **"I don't know."**  
Do NOT use outside knowledge.  

---
Question: {input}
                                                                                
# Retrieved Context Chunks:
<context>
{context}
</context>


---

# Important Rules for Answering:
1. **Strict Priority**: If any chunk’s `source` is **"BNS Delhi police 2nd edition.pdf"**, build the answer **primarily from those BNS chunks**.
   - Use IPC sources (`"ipctextcorpus.txt"`, `"Indian Penal Code Book.pdf"`) **only to fill gaps** not covered by the BNS chunks.
   - If BNS and another source conflict, prefer the BNS wording.
2. **Quoting**: Include as much text as possible from the retrieved context.  
   Wrap quotes in double quotes and cite the source and page/chunk_id like:  
   `[BNS Delhi police 2nd edition.pdf, p. 45]` or `[Indian Penal Code Book.pdf, chunk 12]`.
3. **No Fabrication**: Do not invent or infer sections, punishments, or procedures not present in the retrieved text.  
   If insufficient information exists → reply **"I don't know."**
4. **Clarity**: Explanations must be clear, concise, and tailored to the user’s question.

# Answer Format:
- Direct answer to the user’s question, grounded in the BNS chunks first.  
- Supporting explanation with **quoted or closely paraphrased text** from the context.  
- Citations after each paragraph.  
- If nothing is found: **"I don't know."**



"""
)
from langchain.chains.combine_documents import create_stuff_documents_chain

document_chain=create_stuff_documents_chain(llm,prompt)

retriever=vectordb.as_retriever()

from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)

response=retrieval_chain.invoke({"input":"What is a bail? What is anticipatory bail? How do I get a bail? How much I have to pay for a bail? Is that amount refundable? Can I take bail in advance if I know somebody is going to take a legal action against me (I did not do anything wrong)? How long a bail is valid? Does it have some expiry?"})
response


⚠️ It looks like you upgraded from a version below 0.6 and could benefit from vacuuming your database. Run chromadb utils vacuum --help for more information.
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
}
violations {
}
violations {
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 48
}
].
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
}


In [ ]:
j=0
for i in a:
    response=retrieval_chain.invoke({"input":i})
    df['llm_answer'][j]=response['answer']
    df['contexts'][j]=response['context']
    j+=1

In [ ]:
df


In [ ]:
df.to_csv("llm_answer_lamma3.1.csv")


# deleting db

In [ ]:
!zip -r db.zip ./db

In [ ]:
vectordb.delete_collection()
vectordb.persist()

# delete the directory
!rm -rf db/

# starting again loading db

In [ ]:
!unzip db.zip